# 02 - Data Cleaning
Collapses the 42 years x 6 bands x 3 observation slots into a location-by-year grid (the "cube"), diagnoses contamination, then cleans in five stages.

**Reads:** raw parquet files + `processed/*_px.parquet` (from notebook 01)
**Writes:** `processed/madrid_cube2.pkl`, `processed/amsterdam_cube2.pkl`
**Next:** `03_eda.ipynb`

Sections: A) first-pass cube, B) extreme-value diagnostics, C) why cleaning matters, D) five-stage cleaning, E) save.

## 1. Setup and load

In [1]:
import os
import numpy as np
import pandas as pd

DATA_DIR = "data"
OUT_DIR = "processed"
os.makedirs(OUT_DIR, exist_ok=True)

BANDS = ["Blue", "Green", "Red", "NIR", "SWIR1", "SWIR2"]
KEY = ["px_key", "py_key"]          # this pair identifies a location
YEARS = np.arange(1984, 2026)
print("OK - ready")

OK - ready


In [2]:
madrid = pd.read_parquet(f"{DATA_DIR}/madrid_train.parquet")
amsterdam = pd.read_parquet(f"{DATA_DIR}/amsterdam_data.parquet")
madrid_px = pd.read_parquet(f"{OUT_DIR}/madrid_px.parquet")
amsterdam_px = pd.read_parquet(f"{OUT_DIR}/amsterdam_px.parquet")
print("madrid", madrid.shape, madrid_px.shape)
print("amsterdam", amsterdam.shape, amsterdam_px.shape)

madrid (3172765, 48) (76263, 4)
amsterdam (1062877, 48) (25992, 4)


## A. First-pass processing (before cleaning)
Up to 3 observation slots per year become one averaged value per band, then a location x year grid per band with gaps interpolated. This is the "before cleaning" baseline.

In [3]:
def flatten_observations(df):
    """Up to 3 observation slots per year -> one averaged value per band.

    Only slots flagged valid by qa_valid_N and carrying a physically
    plausible reflectance are averaged. Years with no usable observation
    become NaN and are filled in build_cube().
    """
    out = df[KEY + ["year"]].copy()
    for b in BANDS:
        total = np.zeros(len(df), dtype="float32")
        count = np.zeros(len(df), dtype="float32")
        for n in (1, 2, 3):
            col, qa = f"{b}_{n}", f"qa_valid_{n}"
            if col not in df.columns:
                continue
            v = df[col].to_numpy(dtype="float32")
            if qa in df.columns:
                m = df[qa].fillna(False).to_numpy(dtype=bool)
            else:
                m = np.isfinite(v)
            m = m & np.isfinite(v) & (v > 0) & (v < 65535)
            total += np.where(m, v, 0.0)
            count += m
        out[b] = np.where(count > 0, total / np.maximum(count, 1), np.nan)
    return out
print("OK - defined: flatten_observations")

OK - defined: flatten_observations


In [4]:
def build_cube(flat):
    """One location x year grid per band, cloud gaps filled.

    interpolate(axis=1) fills interior gaps linearly along the time axis;
    limit_direction='both' carries the nearest observed value outward at
    the edges. This matches the starter notebook's behaviour (the
    organisers' video says nearest-neighbour throughout - trust the code).
    """
    cube = {}
    for b in BANDS:
        p = (flat.pivot_table(index=KEY, columns="year", values=b, aggfunc="mean")
                 .reindex(columns=YEARS))
        cube[b] = p.interpolate(axis=1, limit_direction="both")
    return cube
print("OK - defined: build_cube")

OK - defined: build_cube


In [5]:
# ---- no-matplotlib checks ----
BLOCKS = ".:-=+*#@"


def spark(values):
    """42 characters, one per year, 1984 on the left."""
    v = np.asarray(values, dtype="float32")
    v = v[np.isfinite(v)]
    if len(v) == 0:
        return ""
    lo, hi = v.min(), v.max()
    if hi == lo:
        return BLOCKS[0] * len(v)
    idx = ((v - lo) / (hi - lo) * (len(BLOCKS) - 1)).round().astype(int)
    return "".join(BLOCKS[i] for i in idx)


def inspect(cube, px, city, band="NIR"):
    p = cube[band]
    print(f"\n=== {city.upper()} / {band} ===")
    print(f"grid {p.shape}   missing after fill {p.isna().mean().mean():.2%}")

    labels = px["age_class"].reindex(p.index)
    print(f"\nmean {band} trajectory, 1984 -> 2025")
    for c in (1, 2, 3, 4):
        s = p[labels == c].mean()
        print(f"  class {c}  {spark(s)}  {s.min():7.0f} - {s.max():7.0f}")

    # If construction leaves a trace, volatility should rise with recency.
    d = p.diff(axis=1).abs().mean(axis=1)
    print("\nmean |year-on-year change| per class")
    for c in (1, 2, 3, 4):
        print(f"  class {c}  {d[labels == c].mean():8.1f}")


# demo so this cell shows something: a steadily rising signal
print("spark() demo, rising signal 1984 -> 2025:")
print("  " + spark(np.linspace(0, 1, 42)))
print("  '.' = lowest value, '@' = highest value")

spark() demo, rising signal 1984 -> 2025:
  ...::::::------======++++++******######@@@
  '.' = lowest value, '@' = highest value


In [6]:
madrid_flat = flatten_observations(madrid)
madrid_cube = build_cube(madrid_flat)
print("madrid cube built:", madrid_cube["NIR"].shape)

madrid cube built: (76263, 42)


In [7]:
inspect(madrid_cube, madrid_px, "madrid")


=== MADRID / NIR ===
grid (76263, 42)   missing after fill 0.00%

mean NIR trajectory, 1984 -> 2025
  class 1  *=**++++*===--*=@=*++-@++-:.==+@#:=+*:-*++    13453 -   15573
  class 2  @*##*****+++--*=#=*++-#++-:.=-+##:==+:-*+=    13531 -   16160
  class 3  @*@@*####***==#+#+*++-#==-:.=-=#*:==+:-+==    13907 -   16848
  class 4  #+##=+#+*=**=-*+#*##*+@*+=:.==+#*:-=+.:+==    15220 -   17585

mean |year-on-year change| per class
  class 1     726.1
  class 2     800.2
  class 3     873.6
  class 4     998.5


In [8]:
amsterdam_flat = flatten_observations(amsterdam)
amsterdam_cube = build_cube(amsterdam_flat)
print("amsterdam cube built:", amsterdam_cube["NIR"].shape)

amsterdam cube built: (25992, 42)


In [9]:
inspect(amsterdam_cube, amsterdam_px, "amsterdam")


=== AMSTERDAM / NIR ===
grid (25992, 42)   missing after fill 0.00%

mean NIR trajectory, 1984 -> 2025
  class 1  @##+=#-*@=*++-+===@-==*=++===-+.=-+:+*=*-=    11056 -   14463
  class 2  #**+=*:+#-++=-=-==@-==+==+=--:=.=-=:=+=+-=    11966 -   16230
  class 3  ####*#=#@+**+=+===@-=-+===---:-.=--:==-=:-    11809 -   16799
  class 4  ++++++-+*-++=-===+@=+=+===---:-.-:-:--:-::    11811 -   18701

mean |year-on-year change| per class
  class 1    1126.5
  class 2    1274.3
  class 3    1344.5
  class 4    1418.4


## B. Extreme-value diagnostics
The `qa_valid_N` flags miss haze and thin cloud. Contaminated scenes survive as plausible-looking but inflated reflectances. These helpers find them.

In [10]:
def band_distribution(cube, city):
    print(f"\n=== {city.upper()}: value distribution per band ===")
    rows = []
    for b in BANDS:
        v = cube[b].to_numpy().ravel()
        v = v[np.isfinite(v)]
        rows.append({
            "band": b,
            "p01": np.percentile(v, 1),
            "median": np.percentile(v, 50),
            "p99": np.percentile(v, 99),
            "p99.9": np.percentile(v, 99.9),
            "max": v.max(),
            "ratio_max_med": v.max() / np.percentile(v, 50),
        })
    out = pd.DataFrame(rows).set_index("band").round(1)
    print(out)
    print("\nA max far above p99.9 means a few contaminated scenes, not a "
          "gradual tail.")
    return out


def suspect_years(cube, city, n_mad=5.0):
    """Flag years whose city-wide mean sits far from the typical year.

    Uses median absolute deviation, which a handful of bad years cannot
    drag around the way a standard deviation can.
    """
    print(f"\n=== {city.upper()}: suspect years (> {n_mad} MAD from median) ===")
    flagged = {}
    for b in BANDS:
        yearly = cube[b].mean(axis=0)                  # one value per year
        med = yearly.median()
        mad = (yearly - med).abs().median()
        if mad == 0:
            continue
        score = (yearly - med).abs() / mad
        bad = score[score > n_mad].sort_values(ascending=False)
        if len(bad):
            flagged[b] = bad
            for yr, s in bad.items():
                print(f"  {b:6s} {yr}  mean {yearly[yr]:8.0f}  "
                      f"(typical {med:.0f}, {s:.1f} MAD)")
    if not flagged:
        print("  none")
    return flagged
print("OK - defined: band_distribution, suspect_years")

OK - defined: band_distribution, suspect_years


In [11]:
def extreme_locations(cube, city, threshold=25000):
    print(f"\n=== {city.upper()}: locations with values above {threshold:,} ===")
    any_hit = None
    for b in BANDS:
        hit = (cube[b] > threshold)
        n_cells = int(hit.to_numpy().sum())
        n_loc = int(hit.any(axis=1).sum())
        print(f"  {b:6s} {n_cells:8,} cells   {n_loc:6,} locations "
              f"({100*n_loc/len(cube[b]):.2f}%)")
        any_hit = hit.any(axis=1) if any_hit is None else (any_hit | hit.any(axis=1))
    print(f"  ANY    {int(any_hit.sum()):6,} locations affected "
          f"({100*any_hit.mean():.2f}%)")
    return any_hit


def mask_and_refill(flat, blue_max=15000, hard_max=25000):
    """Blank contaminated observations, then re-interpolate.

    Two rules:
      - Blue above blue_max: haze scatters blue light most, so an inflated
        blue band is the clearest signature of atmospheric contamination.
      - Any band above hard_max: implausible for an urban surface.

    A whole year for a location is dropped if either rule fires, because
    contamination affects all bands of that scene together.
    (Exploratory helper - the pipeline below uses the 5-stage clean() instead.)
    """
    f = flat.copy()
    bad = (f["Blue"] > blue_max)
    for b in BANDS:
        bad |= (f[b] > hard_max)
    print(f"masking {int(bad.sum()):,} of {len(f):,} pixel-years "
          f"({100*bad.mean():.2f}%)")
    f.loc[bad, BANDS] = np.nan
    return f
print("OK - defined: extreme_locations, mask_and_refill")

OK - defined: extreme_locations, mask_and_refill


In [12]:
_ = band_distribution(madrid_cube, "madrid")
_ = suspect_years(madrid_cube, "madrid")
print("OK - ready")


=== MADRID: value distribution per band ===
                p01        median           p99         p99.9      max  \
band                                                                     
Blue    8794.400391  10398.500000  15610.299805  28820.400391  64126.0   
Green   9330.000000  11306.700195  17414.400391  24968.000000  62533.5   
Red     9626.000000  12511.200195  19111.900391  32535.000000  64699.0   
NIR    11825.400391  15183.500000  21680.099609  26674.800781  63769.0   
SWIR1  11480.000000  15846.099609  23837.800781  28276.300781  65454.0   
SWIR2  10303.299805  14526.000000  21663.800781  26875.300781  65454.0   

       ratio_max_med  
band                  
Blue             6.2  
Green            5.5  
Red              5.2  
NIR              4.2  
SWIR1            4.1  
SWIR2            4.5  

A max far above p99.9 means a few contaminated scenes, not a gradual tail.

=== MADRID: suspect years (> 5.0 MAD from median) ===
  none
OK - ready


In [13]:
_ = extreme_locations(madrid_cube, "madrid")
print("OK - ready")


=== MADRID: locations with values above 25,000 ===
  Blue      4,036 cells    1,426 locations (1.87%)
  Green     3,171 cells    1,098 locations (1.44%)
  Red       6,387 cells    2,400 locations (3.15%)
  NIR       6,325 cells    2,684 locations (3.52%)
  SWIR1    18,544 cells    7,701 locations (10.10%)
  SWIR2     6,016 cells    1,892 locations (2.48%)
  ANY     9,313 locations affected (12.21%)
OK - ready


In [14]:
_ = band_distribution(amsterdam_cube, "amsterdam")
_ = suspect_years(amsterdam_cube, "amsterdam")
print("OK - ready")


=== AMSTERDAM: value distribution per band ===
                p01        median           p99         p99.9           max  \
band                                                                          
Blue    8233.599609   9209.200195  16307.299805  28757.000000  64126.000000   
Green   8480.000000   9725.200195  17762.199219  28655.400391  62533.500000   
Red     8578.700195  10021.500000  18945.599609  30148.199219  64388.101562   
NIR     9997.900391  13406.700195  21619.699219  26445.800781  60322.000000   
SWIR1  10097.900391  12921.000000  19856.400391  25689.800781  65454.000000   
SWIR2   9353.599609  11551.400391  17963.300781  24771.400391  65454.000000   

       ratio_max_med  
band                  
Blue             7.0  
Green            6.4  
Red              6.4  
NIR              4.5  
SWIR1            5.1  
SWIR2            5.7  

A max far above p99.9 means a few contaminated scenes, not a gradual tail.

=== AMSTERDAM: suspect years (> 5.0 MAD from median) ===
 

In [15]:
_ = extreme_locations(amsterdam_cube, "amsterdam")
print("OK - ready")


=== AMSTERDAM: locations with values above 25,000 ===
  Blue      7,308 cells    7,046 locations (27.11%)
  Green     7,351 cells    7,060 locations (27.16%)
  Red       7,560 cells    7,098 locations (27.31%)
  NIR       2,229 cells    1,556 locations (5.99%)
  SWIR1     1,335 cells      528 locations (2.03%)
  SWIR2     1,031 cells      281 locations (1.08%)
  ANY     7,384 locations affected (28.41%)
OK - ready


## C. Why cleaning matters
Saturation codes near 65535 in the raw Blue slots, and which years they cluster in.

In [16]:
v = amsterdam[["Blue_1","Blue_2","Blue_3"]].max(axis=1)
print(amsterdam.loc[v > 60000, "year"].value_counts().head())

year
1998    11682
2008     7679
2002     7035
1989      982
1990      383
Name: count, dtype: int64


In [17]:
v = madrid[["Blue_1","Blue_2","Blue_3"]].max(axis=1)
print(madrid.loc[v > 60000, "year"].value_counts().head())

year
1995    804
2007    275
2010    241
2006    231
2008    185
Name: count, dtype: int64


## D. Five-stage cleaning
1. saturation codes, filtered at observation-slot level
2. whole-scene contamination, masked per city-year, all bands
3. per-location temporal spikes, judged against the location itself
4. re-interpolate and rebuild the cube (reuses `build_cube` from section A)
5. verify, and report what was removed

Run `diagnose()` first to choose thresholds from the data, then `clean()`.

In [18]:
# Tune these from diagnose() output
HARD_MAX = 40000      # stage 1: above this is a saturation code, not a reading
YEAR_MAD = 10.0       # stage 2: how far a year's city mean may sit from typical
LOC_MAD = 10.0        # stage 3: how far one year may sit from a location's own median
print("OK - ready")

OK - ready


In [19]:
def diagnose(df, city):
    """Where does real data end and artifact begin?"""
    print(f"\n{'='*60}\n{city.upper()}: choosing thresholds\n{'='*60}")

    print("\n-- stage 1: is there a dead zone below the saturation codes? --")
    edges = [25000, 30000, 35000, 40000, 45000, 50000, 55000, 60000, 66000]
    for b in BANDS:
        vals = []
        for n in (1, 2, 3):
            col = f"{b}_{n}"
            if col in df.columns:
                vals.append(df[col].to_numpy(dtype="float32"))
        v = np.concatenate(vals)
        v = v[np.isfinite(v)]
        counts = [int(((v >= lo) & (v < hi)).sum())
                  for lo, hi in zip(edges[:-1], edges[1:])]
        print(f"  {b:6s} " + " ".join(f"{c:>7,}" for c in counts))
    print("  bins:  " + " ".join(f"{lo//1000:>4}k-{hi//1000:<2}k"
                                 for lo, hi in zip(edges[:-1], edges[1:])))
    print("  A run of zeros then a spike at the top = clean cut available.")
print("OK - defined: diagnose")

OK - defined: diagnose


In [22]:
def flatten_clean(df, hard_max=HARD_MAX):
    """Stage 1: average the valid observation slots, rejecting saturation codes.

    The rejection happens per slot, before averaging, so one bad slot
    cannot contaminate a location-year that also has a good slot.
    """
    out = df[KEY + ["year"]].copy()
    rejected = 0
    for b in BANDS:
        total = np.zeros(len(df), dtype="float32")
        count = np.zeros(len(df), dtype="float32")
        for n in (1, 2, 3):
            col, qa = f"{b}_{n}", f"qa_valid_{n}"
            if col not in df.columns:
                continue
            v = df[col].to_numpy(dtype="float32")
            m = (df[qa].fillna(False).to_numpy(dtype=bool)
                 if qa in df.columns else np.isfinite(v))
            ok = np.isfinite(v) & (v > 0) & (v < hard_max)
            rejected += int((m & ~ok).sum())
            m = m & ok
            total += np.where(m, v, 0.0)
            count += m
        out[b] = np.where(count > 0, total / np.maximum(count, 1), np.nan)
    print(f"  stage 1: rejected {rejected:,} saturated slot readings")
    return out
print("OK - defined: flatten_clean")

OK - defined: flatten_clean


In [23]:
def mask_bad_scenes(flat, n_mad=YEAR_MAD):
    """Stage 2: mask every band of a year whose city-wide mean is an outlier.

    Contamination is a property of the scene, so when one band flags,
    the whole year goes. Returns the cleaned frame and the years masked.
    """
    yearly = flat.groupby("year")[BANDS].mean()
    flagged = set()
    for b in BANDS:
        s = yearly[b]
        med = s.median()
        mad = (s - med).abs().median()
        if mad == 0:
            continue
        score = (s - med).abs() / mad
        for yr in score[score > n_mad].index:
            flagged.add(int(yr))
            print(f"  stage 2: {yr} flagged on {b} "
                  f"({score[yr]:.1f} MAD, mean {s[yr]:.0f} vs {med:.0f})")
    if not flagged:
        print("  stage 2: no contaminated scenes")
        return flat, []

    f = flat.copy()
    hit = f["year"].isin(flagged)
    f.loc[hit, BANDS] = np.nan
    print(f"  stage 2: masked years {sorted(flagged)} "
          f"({int(hit.sum()):,} pixel-years, all bands)")
    return f, sorted(flagged)
print("OK - defined: mask_bad_scenes")

OK - defined: mask_bad_scenes


In [24]:
def mask_local_spikes(flat, n_mad=LOC_MAD):
    """Stage 3: mask a value far from that location's own median across years.

    A location that always reads high keeps its values; a location that
    reads high once does not. Identical rule in both cities.
    """
    f = flat.copy()
    total = 0
    for b in BANDS:
        g = f.groupby(KEY)[b]
        med = g.transform("median")
        dev = (f[b] - med).abs()
        mad = dev.groupby([f[KEY[0]], f[KEY[1]]]).transform("median")
        spike = (mad > 0) & (dev / mad.replace(0, np.nan) > n_mad)
        total += int(spike.sum())
        f.loc[spike, b] = np.nan
    print(f"  stage 3: masked {total:,} isolated spikes "
          f"({100*total/(len(f)*len(BANDS)):.3f}% of all values)")
    return f
print("OK - defined: mask_local_spikes")

OK - defined: mask_local_spikes


In [25]:
def verify(cube, px, city, band="NIR"):
    """Stage 5: verify the cleaned cube."""
    p = cube[band]
    print(f"\n-- {city}: after cleaning --")
    print(f"  grid {p.shape}   missing {p.isna().mean().mean():.2%}")

    # suspect years should now be empty
    yearly = p.mean(axis=0)
    med, mad = yearly.median(), (yearly - yearly.median()).abs().median()
    worst = ((yearly - med).abs() / mad).max() if mad else 0
    print(f"  worst remaining year: {worst:.1f} MAD (was 46.4 for Amsterdam Blue)")

    labels = px["age_class"].reindex(p.index)
    d = p.diff(axis=1).abs().mean(axis=1)
    vols = [d[labels == c].mean() for c in (1, 2, 3, 4)]
    print("  volatility by class: " + "  ".join(f"{v:.0f}" for v in vols))
    return vols


def clean(df, px, city):
    print(f"\n{'='*60}\nCLEANING {city.upper()}\n{'='*60}")
    flat = flatten_clean(df)
    flat, bad_years = mask_bad_scenes(flat)
    flat = mask_local_spikes(flat)
    cube = build_cube(flat)
    vols = verify(cube, px, city)
    return flat, cube, vols
print("OK - defined: verify, clean")

OK - defined: verify, clean


### D1. Choose thresholds (diagnose)

In [26]:
diagnose(madrid, "madrid")


MADRID: choosing thresholds

-- stage 1: is there a dead zone below the saturation codes? --
  Blue    23,683  34,827  14,553     812      14       8      13   4,032
  Green   30,616  29,249  14,011   1,821     102      55       7   1,090
  Red     29,706  28,941  16,836   3,437     322      80       7   2,862
  NIR     34,526  27,350  18,106   5,010     693      77      55      33
  SWIR1   72,333   6,237     891     327     141      87      68      90
  SWIR2   17,269   2,801     922     382     158      83      64     123
  bins:    25k-30k   30k-35k   35k-40k   40k-45k   45k-50k   50k-55k   55k-60k   60k-66k
  A run of zeros then a spike at the top = clean cut available.


In [27]:
diagnose(amsterdam, "amsterdam")


AMSTERDAM: choosing thresholds

-- stage 1: is there a dead zone below the saturation codes? --
  Blue    19,627  17,692   3,216     262      77       1       1  28,646
  Green   26,363  12,824  12,307   1,107     214     147      72  14,082
  Red     25,028  14,365  13,777   2,381     206     176     126  14,726
  NIR     21,489  22,176  16,033  10,400     885     189     167     281
  SWIR1   12,937   5,848     599     354     238     195     158     140
  SWIR2    6,887   1,932     483     278     167      58      21      52
  bins:    25k-30k   30k-35k   35k-40k   40k-45k   45k-50k   50k-55k   55k-60k   60k-66k
  A run of zeros then a spike at the top = clean cut available.


### D2. Run the cleaning

In [28]:
madrid_flat2, madrid_cube2, mv = clean(madrid, madrid_px, "madrid")
print("OK - ready")


CLEANING MADRID
  stage 1: rejected 19,025 saturated slot readings
  stage 2: no contaminated scenes
  stage 3: masked 48,447 isolated spikes (0.254% of all values)

-- madrid: after cleaning --
  grid (76263, 42)   missing 0.00%
  worst remaining year: 3.1 MAD (was 46.4 for Amsterdam Blue)
  volatility by class: 700  770  849  969
OK - ready


In [29]:
amsterdam_flat2, amsterdam_cube2, av = clean(amsterdam, amsterdam_px, "amsterdam")
print("OK - ready")


CLEANING AMSTERDAM
  stage 1: rejected 28,906 saturated slot readings
  stage 2: no contaminated scenes
  stage 3: masked 41,318 isolated spikes (0.648% of all values)

-- amsterdam: after cleaning --
  grid (25992, 42)   missing 0.00%
  worst remaining year: 4.3 MAD (was 46.4 for Amsterdam Blue)
  volatility by class: 976  1121  1171  1287
OK - ready


In [30]:
print("=" * 60)
print("DOMAIN GAP: mean volatility, Amsterdam / Madrid")
print(f"  before cleaning: {1290/862:.2f}x")
print(f"  after cleaning:  {np.mean(av)/np.mean(mv):.2f}x")
print("  Lower is better - it means less difference between the two")
print("  cities that has nothing to do with buildings.")

DOMAIN GAP: mean volatility, Amsterdam / Madrid
  before cleaning: 1.50x
  after cleaning:  1.39x
  Lower is better - it means less difference between the two
  cities that has nothing to do with buildings.


## E. Save the cleaned cubes

In [31]:
pd.to_pickle(madrid_cube2, f"{OUT_DIR}/madrid_cube2.pkl")
pd.to_pickle(amsterdam_cube2, f"{OUT_DIR}/amsterdam_cube2.pkl")
print("saved:", sorted(f for f in os.listdir(OUT_DIR) if f.endswith(".pkl")))

saved: ['amsterdam_cube2.pkl', 'madrid_cube2.pkl']
